In [ ]:
data_quality_analysis.ipynb

In [2]:
%pip install great_expectations

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np

# Load the cleaned CSV dataset
df = pd.read_csv("../week2_day4/cleaned_dataset.csv")

# Display first 5 rows
print("Dataset Preview:")
display(df.head())

# Display dataset size
print("Dataset Shape:", df.shape)

# Display column names
print("Column Names:")
print(df.columns.tolist())

Dataset Preview:


,sale_id,sale_date,region,category,product,quantity,sales_amount,profit
0,1,2026-01-05,karnataka,electronics,laptop,2.0,77500.0,10000.0
1,2,2026-01-10,kerala,furniture,chair,5.0,15000.0,3000.0
2,3,2026-02-12,tamil nadu,electronics,mobile,3.0,45000.0,7000.0
3,4,2026-02-15,karnataka,clothing,shirt,9.5,10000.0,2500.0
4,5,2026-03-05,kerala,electronics,tablet,4.0,40000.0,6000.0


Dataset Shape: (120, 8)
Column Names:
['sale_id', 'sale_date', 'region', 'category', 'product', 'quantity', 'sales_amount', 'profit']


In [4]:
# 1. COMPLETENESS
print("Missing values:")
print(df.isnull().sum())

# 2. ACCURACY
print("\nInvalid quantity:", (df["quantity"] <= 0).sum())
print("Invalid sales amount:", (df["sales_amount"] < 0).sum())

# 3. CONSISTENCY
df["sale_date"] = pd.to_datetime(df["sale_date"], errors="coerce")
print("\nInvalid sale dates:", df["sale_date"].isnull().sum())

# 4. TIMELINESS
print("\nFuture sale dates:", (df["sale_date"] > pd.Timestamp.today()).sum())

# 5. UNIQUENESS
print("\nDuplicate sale IDs:", df["sale_id"].duplicated().sum())

# 6. VALIDITY
print("\nUnique regions:", df["region"].unique())
print("Unique categories:", df["category"].unique())

Missing values:
sale_id         0
sale_date       0
region          0
category        0
product         0
quantity        0
sales_amount    0
profit          0
dtype: int64

Invalid quantity: 0
Invalid sales amount: 0

Invalid sale dates: 0

Future sale dates: 0

Duplicate sale IDs: 0

Unique regions: ['karnataka' 'kerala' 'tamil nadu']
Unique categories: ['electronics' 'furniture' 'clothing']


In [5]:
# Calculate quality checks

total_rows = len(df)

quality_results = {
    "Completeness": df.notnull().all(axis=1).sum(),
    "Accuracy": ((df["quantity"] > 0) & (df["sales_amount"] >= 0)).sum(),
    "Consistency": df["sale_date"].notnull().sum(),
    "Timeliness": (df["sale_date"] <= pd.Timestamp.today()).sum(),
    "Uniqueness": df["sale_id"].is_unique * total_rows,
    "Validity": (
        df["region"].notnull() &
        df["category"].notnull()
    ).sum()
}

# Create scorecard
scorecard = pd.DataFrame({
    "Dimension": quality_results.keys(),
    "Passed Rows": quality_results.values()
})

scorecard["Failed Rows"] = total_rows - scorecard["Passed Rows"]
scorecard["Pass Percentage"] = (
    scorecard["Passed Rows"] / total_rows * 100
).round(2)

display(scorecard)

,Dimension,Passed Rows,Failed Rows,Pass Percentage
0,Completeness,120,0,100.0
1,Accuracy,120,0,100.0
2,Consistency,120,0,100.0
3,Timeliness,120,0,100.0
4,Uniqueness,120,0,100.0
5,Validity,120,0,100.0


In [6]:
import great_expectations as gx

print("Great Expectations Version:", gx.__version__)
print("Great Expectations imported successfully!")

Great Expectations Version: 1.25.0
Great Expectations imported successfully!


In [7]:
# Create Great Expectations context
context = gx.get_context()

# Create a Pandas data source
data_source = context.data_sources.add_pandas("sales_data_source")

# Create a DataFrame asset
data_asset = data_source.add_dataframe_asset(
    name="sales_data_asset"
)

# Create a batch definition
batch_definition = data_asset.add_batch_definition_whole_dataframe(
    "sales_batch"
)

# Connect our dataset
batch = batch_definition.get_batch(
    batch_parameters={"dataframe": df}
)

print("Dataset connected to Great Expectations successfully!")
print("Number of rows:", len(df))

Dataset connected to Great Expectations successfully!
Number of rows: 120


In [8]:
# Check that sale_id values are not missing
expectation = gx.expectations.ExpectColumnValuesToNotBeNull(
    column="sale_id"
)

# Run validation
validation_result = batch.validate(
    expectation
)

print("Validation Result:")
print(validation_result)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Validation Result:
{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_not_be_null",
    "kwargs": {
      "batch_id": "sales_data_source-sales_data_asset",
      "column": "sale_id"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 120,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "partial_unexpected_counts": [],
    "partial_unexpected_index_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}


In [9]:
# Create 8 data quality expectations

expectations = [
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="sale_id"
    ),
    gx.expectations.ExpectColumnValuesToBeUnique(
        column="sale_id"
    ),
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="sale_date"
    ),
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="quantity",
        min_value=1
    ),
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="sales_amount",
        min_value=0
    ),
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="region"
    ),
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="category"
    ),
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="product"
    )
]

print("Total expectations created:", len(expectations))

Total expectations created: 8


In [10]:
# Run all 8 expectations
validation_results = []

for i, expectation in enumerate(expectations, start=1):
    result = batch.validate(expectation)
    validation_results.append(result)

    print(
        f"Check {i}:",
        "PASSED" if result.success else "FAILED"
    )

# Count results
passed = sum(result.success for result in validation_results)
failed = len(validation_results) - passed

print("\nVALIDATION SUMMARY")
print("Total Checks:", len(validation_results))
print("Passed:", passed)
print("Failed:", failed)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Check 1: PASSED


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Check 2: PASSED


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Check 3: PASSED


... (6 outputs left out)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Check 7: PASSED


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Check 8: PASSED

VALIDATION SUMMARY
Total Checks: 8
Passed: 8
Failed: 0


In [11]:
# Make a copy of the original dataset
df_bad = df.copy()

# Error 1: Insert a missing sale_id
df_bad.loc[0, "sale_id"] = None

# Error 2: Insert a duplicate sale_id
df_bad.loc[1, "sale_id"] = df_bad.loc[2, "sale_id"]

# Error 3: Insert an invalid quantity
df_bad.loc[3, "quantity"] = -10

print("Three errors successfully injected!")

display(df_bad.head())

Three errors successfully injected!


,sale_id,sale_date,region,category,product,quantity,sales_amount,profit
0,NaN,2026-01-05,karnataka,electronics,laptop,2.0,77500.0,10000.0
1,3.0,2026-01-10,kerala,furniture,chair,5.0,15000.0,3000.0
2,3.0,2026-02-12,tamil nadu,electronics,mobile,3.0,45000.0,7000.0
3,4.0,2026-02-15,karnataka,clothing,shirt,-10.0,10000.0,2500.0
4,5.0,2026-03-05,kerala,electronics,tablet,4.0,40000.0,6000.0


In [12]:
# Connect corrupted dataset to Great Expectations
bad_batch = batch_definition.get_batch(
    batch_parameters={"dataframe": df_bad}
)

# Check 1: Missing sale IDs
result1 = bad_batch.validate(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="sale_id"
    )
)

# Check 2: Duplicate sale IDs
result2 = bad_batch.validate(
    gx.expectations.ExpectColumnValuesToBeUnique(
        column="sale_id"
    )
)

# Check 3: Negative quantities
result3 = bad_batch.validate(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="quantity",
        min_value=1
    )
)

# Display results
print("Missing sale ID detected:", not result1.success)
print("Duplicate sale ID detected:", not result2.success)
print("Invalid quantity detected:", not result3.success)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Missing sale ID detected: True
Duplicate sale ID detected: True
Invalid quantity detected: True


In [13]:
# Create validation report data
check_names = [
    "Sale ID is not null",
    "Sale ID is unique",
    "Sale date is not null",
    "Quantity is at least 1",
    "Sales amount is not negative",
    "Region is not null",
    "Category is not null",
    "Product is not null"
]

report_df = pd.DataFrame({
    "Quality Check": check_names,
    "Result": [
        "PASSED" if result.success else "FAILED"
        for result in validation_results
    ]
})

# Create HTML report
html_report = """
<h1>Great Expectations - Data Quality Report</h1>
<p>Dataset: Week 2 Sales Dataset</p>
<p>Total Records: 120</p>
""" + report_df.to_html(index=False)

with open("great_expectations_report.html", "w", encoding="utf-8") as file:
    file.write(html_report)

print("HTML report created successfully!")
display(report_df)

HTML report created successfully!


,Quality Check,Result
0,Sale ID is not null,PASSED
1,Sale ID is unique,PASSED
2,Sale date is not null,PASSED
3,Quantity is at least 1,PASSED
4,Sales amount is not negative,PASSED
5,Region is not null,PASSED
6,Category is not null,PASSED
7,Product is not null,PASSED


In [14]:
%%writefile data-contract-sales.yaml

dataContractSpecification: "0.9.3"

id: sales-data-v1

info:
  title: Sales Data Contract
  owner: Data Producer
  consumer: BI Dashboard Team
  description: Data quality agreement for sales dataset

models:
  sales:
    fields:
      sale_id:
        type: string
        required: true
        unique: true
        classification: internal

      sale_date:
        type: date
        required: true
        classification: internal

      region:
        type: string
        required: true
        classification: internal

      category:
        type: string
        required: true
        classification: internal

      product:
        type: string
        required: true
        classification: internal

      quantity:
        type: integer
        required: true
        minimum: 1

      sales_amount:
        type: number
        required: true
        minimum: 0

      profit:
        type: number
        required: true

quality:
  freshness:
    maximum_delay: "2 hours"
  uniqueness:
    field: sale_id
  completeness:
    required_fields:
      - sale_id
      - sale_date
      - region
      - category
      - product

privacy:
  pii_fields: []
  classification: internal

sla:
  availability: "99.5%"
  freshness: "Within 2 hours"

Writing data-contract-sales.yaml


In [15]:
# Classify dataset columns

classification = pd.DataFrame({
    "Column": df.columns,
    "Data Classification": [
        "Internal",
        "Internal",
        "Internal",
        "Internal",
        "Internal",
        "Internal",
        "Confidential",
        "Confidential"
    ],
    "PII": [
        "No", "No", "No", "No",
        "No", "No", "No", "No"
    ]
})

display(classification)

,Column,Data Classification,PII
0,sale_id,Internal,No
1,sale_date,Internal,No
2,region,Internal,No
3,category,Internal,No
4,product,Internal,No
5,quantity,Internal,No
6,sales_amount,Confidential,No
7,profit,Confidential,No


In [16]:
# Create a one-page data quality scorecard

scorecard_content = """# Data Quality Scorecard

## Dataset Information
- Dataset: Week 2 Sales Dataset
- Total Records: 120
- Total Columns: 8
- Producer: Data Analyst
- Consumer: BI Dashboard Team

## Six DAMA Data Quality Dimensions

"""

# Add actual scores from our scorecard
for _, row in scorecard.iterrows():
    scorecard_content += (
        f"| {row['Dimension']} | {row['Passed Rows']} | "
        f"{row['Failed Rows']} | {row['Pass Percentage']}% |\n"
    ) if False else ""

scorecard_content += """| Dimension | Passed Rows | Failed Rows | Pass Percentage |
|---|---:|---:|---:|
"""

for _, row in scorecard.iterrows():
    scorecard_content += (
        f"| {row['Dimension']} | {row['Passed Rows']} | "
        f"{row['Failed Rows']} | {row['Pass Percentage']}% |\n"
    )

scorecard_content += """
## Improvement Recommendations

1. Completeness: Fill or investigate missing required values.
2. Accuracy: Validate quantity and sales amount against business rules.
3. Consistency: Standardise date formats and check related records.
4. Timeliness: Monitor data arrival against the two-hour SLA.
5. Uniqueness: Prevent duplicate sale IDs.
6. Validity: Validate region, category and product values against approved lists.

## Privacy and Security
- Restrict access to confidential sales and profit information.
- Review actual data values for possible personal information.
- Mask or tokenise personal identifiers if they are introduced.
- Follow applicable data protection requirements.

## Conclusion
Regular automated quality checks help maintain reliable sales data
for business reporting and BI dashboards.
"""

with open("data_quality_scorecard.md", "w", encoding="utf-8") as file:
    file.write(scorecard_content)

print("Data quality scorecard created successfully!")

Data quality scorecard created successfully!


In [17]:
import os

files = [
    "data_quality_analysis.ipynb",
    "great_expectations_report.html",
    "data-contract-sales.yaml",
    "data_quality_scorecard.md"
]

print("WEEK 2 DAY 6 - DELIVERABLE CHECK")
print("--------------------------------")

for file in files:
    if os.path.exists(file):
        print(file, "- FOUND")
    else:
        print(file, "- NOT FOUND")

WEEK 2 DAY 6 - DELIVERABLE CHECK
--------------------------------
data_quality_analysis.ipynb - NOT FOUND
great_expectations_report.html - FOUND
data-contract-sales.yaml - FOUND
data_quality_scorecard.md - FOUND
